# Intelligentes Bestellformular

In dieser Übung wollen wir ein lernendes Bestellformular für einen fiktiven Webshop programmieren.
Wenn Sie die beiden Zellen unten ausführst, sehen Sie ein simples Bestellformular.

Wir wollen dafür sorgen, dass die Liste mit Artikeln intelligent sortiert wird. Wenn zum Beispiel häufig eine Zahnbürste zusammen mit Zahnpasta gekauft wird, dann soll Zahnpasta oben in der Liste kommen, wenn wir schon eine Zahnbürste im Warenkorb haben.

Für so eine Anwendung ist Naive Bayes eine sinnvolle Wahl. Zum einen können wir die Parameter leicht im laufenden Betrieb updaten. So kann das System sich schnell anpassen, wenn sich das Kaufverhalten ändert oder neue Artikel dazukommen. Zum anderen müssen in einem solchen Fall oft sehr unterschiedliche Komponenten zusammenarbeiten (zum Beispiel ein TypeScript Frontend und ein Java Backend) - da ist es ein Vorteil, wenn man den Classifier einfach selber implementieren kann und nicht von einem Framework abhängig ist.

Für jeden Artikel, der noch nicht im Warenkorb ist, wollen wir die Wahrscheinlichkeit berechnen, dass dieser noch in den Warenkorb kommt. Dann können wir anschließend die Artikelliste nach dieser Wahrscheinlichkeit sortieren und den wahrscheinlichsten Artikel ganz oben einsortieren.

Für jeden Artikel brauchen wir einen eigenen Classifier. Dieser soll dann $P(y | x)$ vorhersagen. Hier ist $y$ der Artikel aus der Auswahlliste. $x$ enthält einen Eintrag für jeden Artikel (der nicht $y$ ist), mit $x_\text{artikel}=1$, falls der Artikel im Warenkorb ist und $x_\text{artikel}=0$ andernfalls.

Die Naive Bayes Formel sagt $P(y | x) = \frac{P(x | y) \cdot P(y)}{P(x)}$. Wir wollen nach dieser Wahrscheinlichkeit sortieren, wobei wir $P(y | x)$ für alle $y$ bestimmen müssen, die noch nicht im Warenkorb sind. Da $P(x)$ in jedem Fall gleich ist (zumindest nahezu), können wir das auch einfach weg lassen und nach $P(x | y) \cdot P(y)$ sortieren (in dem Naive Bayes Setting kommt es fast nie auf diesen Nenner an, da einen meistens nicht die absoluten Wahrscheinlichkeiten interessieren.

Wir gehen bei Naive Bayes davon aus, dass die $P(x_\text{article})$ unabhängig sind, daher gilt
$$P(x | y) = \prod_\text{article} P(x_\text{article} | y)$$

$P(y=1)$ gibt nun an, wie wahrscheinlich der Artikel $y$ in einem beliebigen Wahrenkorb vorkommt, also die Anzahl an Warenkörben mit $y$ geteilt durch die Anzahl aller Warenkörbe.

$P(x_\text{article} = 1 | y)$ ist die Wahrscheinlichkeit, dass $\text{article}$ im Warenkorb ist, gegeben, dass $y$ im Warenkorb ist. Das ist die Anzahl an malen, dass $\text{article}$ zusammen mit $y$ in einem Warenkorb war, geteilt durch die Anzahl an Warenkörben mit $y$. Wenn $\text{article}$ nicht im Warenkorb ist, wollen wir natürlich stattdessen $P(x_\text{article} = 0 | y) = 1 - P(x_\text{article} = 1 | y)$ verwenden.

Beachten Sie, dass immer $P(y | y) = 1$ (alte Bauernregel: Wenn $y$ im Warenkorb ist, dann ist $y$ im Warenkorb) - wir können also diesen Faktor in dem Produkt weglassen, wenn wir über alle Artikel iterieren.

Nun braucht es noch Laplace Smoothing: Wenn wir zum Beispiel $P(x_\text{article} | y) = \frac{a}{b}$ berechnen wollen (wobei $a$ die Anzahl Warenkörbe mit $\text{article}$ und $y$ und $b$ die Anzahl Warenkörbe mit $y$ ist), dann berechnen wir stattdessen $P(x_\text{article} | y) = \frac{a+1}{b+2}$.

## Optional

Wie könnte man das Modell modifizieren, so dass die Berechnung beim Sortieren der Artikel noch einfacher wird?

In [ ]:
# Hier müssen Sie die Naive Bayes Logik implementieren
import numpy as np
class ArticleSorter:
    def __init__(self, all_articles):
        # Input sind alle Artikel, die im Webshop potentiell gekauft werden können.
        self.all_articles = all_articles
        self.matrix = np.zeros((len(all_articles), len(all_articles)))
        self.n_carts = 0
    
    def sort_articles(self, articles, basket):
        if self.n_carts == 0:
            return sorted(articles)
        
        prob_array = []
        # Input sind die Artikel, die noch zur Auswahl stehen und die Artikel, die schon Warenkorb sind
        for article in articles:
            y_idx = self.all_articles.index(article)
            anzahl_carts_mit_y = self.matrix[y_idx][y_idx] 
            p_y = anzahl_carts_mit_y/self.n_carts # für jedes produkt, das zum auswahl steht kalkuliere wie hoch ist die wkeit, dass dieses Produkt überhaupt in einem Warenkorb vorkommt
            p_x_y_total = 1

            for x_article in self.all_articles:
                if x_article == article:
                    continue
                x_idx = self.all_articles.index(x_article)
                a = self.matrix[y_idx][x_idx] # anzahl zusammen x,y
                b = anzahl_carts_mit_y
                p_x1_y = (a+1)/(b+2) # laplace smoothing, (vermeiden wkeit 0) wenn das nicht gemacht würde, könnten wir bei der multiplikation mit 0 den ganzen produkt ruinieren

                if x_article in basket:
                    p = p_x1_y
                else:
                    p = 1-p_x1_y
                
                p_x_y_total *= p
            
            score = p_y * p_x_y_total
            prob_array.append((article, score))
            #es ist nötig ein algorithmus zu neh,en, damit wir keine fehler erhalten

        prob_array.sort(key=lambda pair: pair[1], reverse=True) # default from small to big but we need in reverse order

        return [article for article, score in prob_array]
    
    def checkout_callback(self, basket):
        # Wird mit dem fertigen Warenkorb aufgerufen, wenn dieser ausgecheckt wird
        for y_idx in range(len(basket)):
            article_y = basket[y_idx]
            real_y_idx = self.all_articles.index(article_y)
            for x_idx in range(len(basket)):
                article_x = basket[x_idx]
                real_x_idx = self.all_articles.index(article_x)
                self.matrix[real_y_idx][real_x_idx] = self.matrix[y_idx][x_idx] + 1
        
        self.n_carts +=1



In [ ]:
import ipywidgets as widgets

# Das ist unser Webshop - den Code hier brauchen Sie nicht anzupassen
class ShoppingSite:
    def __init__(self, items):
        self.article_sorter = ArticleSorter(items)
        
        self.items = set(items)
        self.out = widgets.Output(layout={'border': '1px solid black'})
        self.selection = widgets.Dropdown(
            description='Artikel:'
        )
        self.add_btn = widgets.Button(
            description='In den Warenkorb',
            button_style='info',
            icon='plus'
        )
        self.add_btn.on_click(self.add_to_basket)
        
        self.checkout_btn = widgets.Button(
            description='Einkauf abschließen',
            button_style='info'
        )
        self.checkout_btn.on_click(self.checkout)
        
        self.site = widgets.VBox([self.out, widgets.HBox([self.selection, self.add_btn]), self.checkout_btn])

        self.reset()
    
    def update_selection(self):
        with self.out:
            remaining = self.items - set(self.basket)

            self.selection.options = self.article_sorter.sort_articles(remaining, self.basket)
            if len(remaining) == 0:
                self.selection.disabled = True
                self.add_btn.disabled = True
            else:
                self.selection.value = self.selection.options[0]

                
    def reset(self):
        self.selection.disabled = False
        self.add_btn.disabled = False
        self.out.clear_output()
        with self.out:
            print("Warenkorb:")
        self.basket = []
        self.update_selection()
        
                
    def show_site(self):
        display(self.site)

    def add_to_basket(self, _):
        item = self.selection.value
        self.basket.append(item)
        with self.out:
            print(f"- {item}")
        
        self.update_selection()
    
    def checkout(self, _):
        self.article_sorter.checkout_callback(self.basket)
        print(self.article_sorter.matrix)
        self.reset()
        

ShoppingSite([
    "Apfel", "Banane", "Kirsche", "Zitrone", "Toasbrot", "Jogurt", "Zahnseide",
    "Zahnbürste", "Zahnpasta", "Toaster", "Föhn", "Duschgel", "Seife", "Kornflakes"
    ]).show_site()